# Import Package

In [ ]:
import numpy as np
import numpy.random as npr
import pandas as pd
import os
os.environ.setdefault('MPLCONFIGDIR', 'data/.matplotlib')
from scipy.stats import invwishart, multivariate_normal, cramervonmises, norm, chi2

import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import datetime
from scipy.linalg import sqrtm
import time
sns.set(font="sans-serif")
import difflib
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

from numpy.linalg import inv, cholesky


# Loading Data

In [ ]:
df_click = pd.read_parquet('data/browsing_sessions.parquet') # synthetic browsing data


In [ ]:
#browsing.to_csv('browsing_data.csv')


In [ ]:
df_click['cat'] = df_click['category'].apply(lambda x :int(x))
top10 =[5, 9, 19, 11, 20, 23, 6, 21, 25, 22]
df_click = df_click[df_click['cat'].isin(top10)]


In [ ]:
map_dict = {key: value for value, key in enumerate(top10)}
df_click['cat'] = df_click['cat'].apply(lambda x: map_dict[x])


In [ ]:
map_dict


In [ ]:
L1_dict = {5: 'Toys', 6: 'Books', 9: 'Home', 11: 'Electronics', 19: 'Vehicles', 20: 'Travel', 21: 'Mobile', 22: 'Gaming', 23: 'Appliances', 25: 'Lifestyle'}


In [ ]:
df_click.head()


In [ ]:
df = df_click[['ID','order','session_id','cat']]


In [ ]:
df.columns=['ID','order','session_id','c']


In [ ]:
df


In [ ]:
train_data = df_click[df_click['time']<'2022-02-01']
test_data = df_click[df_click['time']>='2022-02-01']
train_data = train_data[train_data['ID'].isin(set(test_data['ID']).intersection(set(train_data['ID'])))]
test_data = test_data[test_data['ID'].isin(set(test_data['ID']).intersection(set(train_data['ID'])))]


In [ ]:
len(train_data)


In [ ]:
len(test_data)


In [ ]:
mcmc = train_data[['ID','session_id','event','cat']]
mcmc = mcmc.sort_values(['ID','session_id']).reset_index(drop=True)
browsing = mcmc.groupby(['ID','session_id'])['cat'].apply(list).reset_index()

mcmc = pd.merge(mcmc ,browsing , on=['ID','session_id'], how='left')
mcmc.columns = ['ID', 'session_id', 'event', 'choice', 'cat']


In [ ]:
test_data = test_data[['ID','session_id','event','cat']]
test_data = test_data.sort_values(['ID','session_id']).reset_index(drop=True)
test_data = test_data.groupby(['ID','session_id'])['cat'].apply(list).reset_index()


In [ ]:
#reset order
b = mcmc.groupby(['ID','session_id']).agg({'event':'sum'}).reset_index()
order=[]

for i in b['event']:
    for o in range(i):
        order.append(o)
        
mcmc['order'] = order


# Transition Matrix

In [ ]:
def trans_x(cate_list, n_cate=10):
    store = np.zeros((n_cate,n_cate))
    for i,j in zip(cate_list[:len(cate_list)-1], cate_list[1:]):
        store[j][i]+=1
    return store

trans = sum(mcmc['cat'].apply(lambda x: trans_x(x)))
trans_sum = np.sum(trans,axis=0)
trans = np.divide(trans, trans_sum, out=np.zeros_like(trans), where=trans_sum!=0)
trans = trans.transpose()

plt.figure(figsize=(18,12))
sns.set_context("paper", rc={"font.size":14})  
sns.heatmap(trans,annot=True, fmt=".2f",cmap="YlGnBu")
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.title('transition matrix',fontsize=16)


# Switching Matrix 
$\large \left\{  \begin{matrix} D(k',k) &=& 1-T(k',k)\\
D(k,k)&=&0 \end{matrix} \right. $  

In [ ]:
switching_cost = 1-trans
np.fill_diagonal(switching_cost,0)

plt.figure(figsize=(18,12))
sns.set_context("paper", rc={"font.size":14})  
sns.heatmap(switching_cost,annot=True, fmt=".2f",cmap="YlGnBu")
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.title('switching_cost',fontsize=16)


# Calculate x

In [ ]:
n_cate = 10
def cal_x(g_list,n_cate):
    x_list = []
    for i in g_list:
        x = np.zeros((len(i)+1,n_cate))
        for o, choice in enumerate(i):
            x[o+1:][:,choice] +=1
        x_list.append(x)
    return x_list
browsing['x'] = cal_x(browsing['cat'],n_cate)
browsing['choice'] = browsing['x'].apply(lambda x: x[1:]-x[:-1])


In [ ]:
browsing['x_sum'] = browsing['choice'].apply(lambda x: np.sum(x,axis=0))


In [ ]:
test_data['x'] = cal_x(test_data['cat'],n_cate)
test_data['choice'] = test_data['x'].apply(lambda x: x[1:]-x[:-1])


# Fuction

## Sample $\small a_u, s_u, C^{swi}_{u}$
$a_u \sim LogNormal(\theta_a, \Sigma_a)$   
$s_u \sim LogNormal(\theta_s, \sigma_s)$   
$C^{swi}_{u} \sim LogNormal(\theta_{switch}, \sigma_{switch})$

In [ ]:
def sample_au(ID,a_mean,a_sigma,eps=1e-12):
    mydict = {key : np.maximum(np.exp(np.random.multivariate_normal(a_mean,a_sigma)),eps)  for key in ID}
    return mydict

def sample_su(ID, s_mean, s_sigma,eps=1e-12):
    # Shifted lognormal keeps s_u strictly greater than 1.
    mydict = {key : 1 + max(float(np.random.lognormal(s_mean,s_sigma)),eps) for key in ID}
    return mydict

def sample_swi(ID, swi_mean, swi_sigma,eps=1e-12):
    mydict = {key : max(float(np.random.lognormal(swi_mean,swi_sigma)),eps) for key in ID}
    return mydict


## Sample $a_u^*,C_u^{switch*}$
$a_u^* \sim LogNormal(log(a_u), \bar \sigma_a^2)$   
$C_u^{switch*} \sim LogNormal(log(C_u^{switch}), \bar \sigma_{switch}^2)$

In [ ]:
def sample_au_star(au,au_std,eps=1e-12):
    return {cid:np.maximum(np.exp(np.random.multivariate_normal(np.log(np.maximum(values,eps)),au_std)),eps) for cid, values in zip(au.keys(),au.values())}

def sample_su_star(su,su_std,eps=1e-12):
    # Random walk is applied to z_u = s_u - 1, then shifted back.
    return {cid : 1 + max(float(np.random.lognormal(np.log(max(values-1,eps)), su_std)),eps) for cid, values in zip(su.keys(),su.values())}

def sample_swi_star(swi,swi_std,eps=1e-12):
    return {cid : max(float(np.random.lognormal(np.log(max(values,eps)), swi_std)),eps) for cid, values in zip(swi.keys(),swi.values())}


## Marginal utility
$\large \Delta U(k;0) =a_{ku}^{\frac{s_u}{s_u-1}}$   
$\large \Delta U(k;x^{t}_{u}) = (\sum\limits_{k'=1}^{K}a_{k'u}x^{t^{\frac{s_u-1}{s_u}}}_{k'u})^{\frac{1}{s_u-1}}a_{ku}x_{ku}^{t^{-\frac{1}{s_u}}} -C^{switch}_{u}D(k',k)$   

In [ ]:
def cal_mu(ID,x,choice,au,su,swi,switching_cost):
    mu_list=[]
    for cid, x_t, c in zip(ID,x,choice):
        power1 = su[cid] / (su[cid]-1)
        power2 = 1 / (su[cid]-1)
        power3 = -1 / su[cid]
        
        mu = np.zeros((x_t.shape))
        mu[0] = au[cid]**power1
        mu[1:] = np.sum(au[cid]*x_t[1:]**power1,axis=1)**power2*np.identity(x_t.shape[0]-1) *np.asmatrix(au[cid]*((x_t[1:]+1)**power3)) - swi[cid]*np.asmatrix(c)*switching_cost
        mu_list.append(mu)
    return mu_list


## Calculate probability
pdf of LogNormal :$\large f_X(x) =  \frac{1}{\sqrt{(2\pi)^{n} det\Sigma}\prod_{j=1}^{n}x_{j}}e^{-\frac{1}{2}(ln(x)-\mu)^{t}\Sigma^{-1}(ln(x)-\mu)} $   

def log_normal_pdf(x,mu,sigma,n):
    return (2*np.pi)**(-n/2) * np.linalg.det(sigma)**(-1/2) * np.prod(x)**(-1) * np.exp(-(1/2)*(np.mat(np.log(x)-mu)*np.linalg.inv(sigma)*np.transpose(np.mat(np.log(x)-mu))))

In [ ]:
def log_normal_pdf(x,mu,sigma,n):
    return np.exp(-(n/2)*np.log(2*np.pi) - (1/2)*np.log(np.linalg.det(sigma)) - np.sum(np.log(x)) - (1/2)*(np.asmatrix(np.log(x)-mu)*np.linalg.inv(sigma)*np.transpose(np.asmatrix(np.log(x)-mu))))

def shifted_log_normal_pdf(x,mu,sigma,n,shift=1,eps=1e-12):
    return log_normal_pdf(np.maximum(x-shift,eps),mu,sigma,n)


## Calculate Multinomial Logistic
$\Large P(Y_i=j) = \frac {e^{\Delta U_{i}(j;x^t_u)}}{1+\sum_{k=1}^{K}e^{\Delta U_{i}(k;x^t_u)}}$

In [ ]:
def multi_logistic(x):
    return np.array(np.asmatrix(np.exp(x))/np.asmatrix(1+np.sum(np.exp(x),axis=1).reshape(x.shape[0],1)))


In [ ]:
def rank_array(x):
    temp = x.argsort()
    ranks = np.empty_like(x)
    ranks[temp] = np.arange(len(x))
    return ranks


In [ ]:
def stable_softmax(x):
    # Multinomial logit with an outside/stop option whose utility is normalized to 0.
    x = np.asarray(x,dtype=float)
    row_max = np.maximum(np.max(x, axis=1).reshape(x.shape[0],1), 0)
    exp_x = np.exp(x-row_max)
    exp_outside = np.exp(-row_max)
    return exp_x/(exp_outside+np.sum(exp_x, axis=1).reshape(x.shape[0],1))


In [ ]:
def invwishartrand_prec(nu,phi):
    return inv(wishartrand(nu,phi))

def invwishartrand(nu, phi):
    return inv(wishartrand(nu, inv(phi)))

def wishartrand(nu, phi):
    dim = phi.shape[0]
    chol = cholesky(phi)
    #nu = nu+dim - 1
    #nu = nu + 1 - np.arange(1,dim+1)
    foo = np.zeros((dim,dim))
    
    for i in range(dim):
        for j in range(i+1):
            if i == j:
                foo[i,j] = np.sqrt(chi2.rvs(nu-(i+1)+1))
            else:
                foo[i,j]  = npr.normal(0,1)
    return np.dot(chol, np.dot(foo, np.dot(foo.T, chol.T)))


In [ ]:
def calc_likelihood(choice,prob,eps=1e-12):
    choice = np.asarray(choice,dtype=float)
    prob = np.asarray(prob,dtype=float)
    selected_prob = np.sum(choice*prob[:-1],axis=1)
    stop_prob = 1-np.sum(prob[-1])
    return float(np.prod(np.clip(selected_prob,eps,1))*np.clip(stop_prob,eps,1))


## Initialize parameter

In [ ]:
import random
random.seed(123)
simulate = browsing
simulate['len'] = simulate['cat'].apply(lambda x : len(x))
simulate = simulate[simulate['ID'].isin(list(simulate[simulate['len']>100]['ID']))==0]
simulate['switch_cost'] = simulate['choice'].apply(lambda x: np.append(np.zeros((1,10)),x,axis=0))
simulate['switch_cost'] = simulate['switch_cost'].apply(lambda x: np.asmatrix(x)*switching_cost)
user_list = random.sample(sorted(set(simulate['ID'])), 5000)
simulate = simulate[simulate['ID'].isin(user_list)]
n_user = len(user_list)


In [ ]:
simulate['g_list'] = simulate['cat'].apply(lambda x: str(x))
count_same = simulate.groupby(['ID','g_list']).size().reset_index(name='count')
simulate = pd.merge(count_same,simulate,how='right',on=['ID','g_list'])
simulate = simulate.drop_duplicates(['ID','g_list']).drop('g_list',axis=1).reset_index(drop=True)


In [ ]:
len(set(simulate['ID']))


In [ ]:
#np.random.seed(123)
ID = list(simulate['ID'].drop_duplicates())
n_user = len(ID)

simulate['x_sum']= simulate['x'].apply(lambda x : x[-1])
test_df = simulate.groupby('ID')['x_sum'].apply(np.sum).reset_index()
test_df['x_sum'] = test_df['x_sum'].apply(lambda x: rank_array(x+np.random.rand(10))+1)

corr = pd.read_csv('data/corr.csv')
corr = np.array(corr.iloc[0:,1:])

a_mean_bar_init = np.asmatrix(np.zeros(10))
a_mean_sigma_init = 1*np.identity(n_cate) # was 1e5
a_sigma_S_init = np.identity(n_cate)
a_df_init = n_cate

a_mean = multivariate_normal.rvs(np.array(a_mean_bar_init).reshape(n_cate),a_mean_sigma_init) 
a_sigma = invwishartrand_prec(a_df_init,np.linalg.inv(a_sigma_S_init))

s_mean = 5
s_sigma = 1

swi_mean_bar_init = 0
swi_mean_sigma_init = 1 # was 1e5
swi_sigma_S_init = 1*np.identity(1)
swi_df_init = 1

swi_mean = np.random.normal(swi_mean_bar_init,swi_mean_sigma_init)
swi_sigma = float(np.asarray(invwishartrand_prec(swi_df_init,np.linalg.inv(swi_sigma_S_init))).ravel()[0])

#test_df.index=test_df['ID']
#au = test_df['x_sum'].to_dict()
au = sample_au(ID,a_mean,a_sigma)
su = sample_su(ID,s_mean,s_sigma)
swi = sample_swi(ID,swi_mean,swi_sigma)

simulate['au'] = simulate['ID'].apply(lambda x: au[x])
simulate['su'] = simulate['ID'].apply(lambda x: su[x])
simulate['swi'] = simulate['ID'].apply(lambda x: swi[x])
simulate['c'] = simulate['swi']*simulate['switch_cost']
#test_data['len'] = test_data['cat'].apply(lambda x : len(x))
train_accuracy = []
test_accuary = []

collect_au = np.zeros((1,n_cate,n_user))[1:]
collect_a_mean = np.zeros((1,n_cate))[1:]
collect_a_sigma = np.zeros((1,n_cate,n_cate))[1:]
accept_a = np.zeros((1,n_user))[1:]

collect_su = np.zeros((1,n_user))[1:]
accept_s = np.zeros((1,n_user))[1:]

collect_swi = np.zeros((1,n_user))[1:]
collect_swi_mean = np.zeros(1)[1:]
collect_swi_sigma = np.zeros(1)[1:]
accept_swi = np.zeros((1,n_user))[1:]


$\theta_a \sim MVN(\theta_{\theta_{a}}, \Sigma_{\theta_{a}})$   
$\Sigma_a \sim InverseWishart(S_{a}^{-1},v_{a})$   

$\theta_{\theta_{a}} = \Sigma_{\theta_{a}}'((\sum_{u=1}^{U}log(a_u))'\Sigma_{a}^{-1}+\bar\theta_{\theta_{a}}\bar\Sigma_{\theta_{a}}^{-1})'$   
$\Sigma_{\theta_{a}} = (U\Sigma_{a}^{-1}+\bar\Sigma_{\theta_{a}}^{-1})^{-1}$   
$S_{a} = \sum_{u=1}^{U}(log(a_{u})-\theta_{a})(log(a_{u})-\theta_{a})'+\bar S_{a}$   
$v_a = U + \bar v_a$

In [ ]:
#new
def cal_mu(au,su,c,x):
    data = pd.DataFrame()
    data['first'] = au**(su / (su-1))
    data['first'] = data['first'].apply(lambda x: x.reshape(1,10))
    data['mu'] = au*x**(su / (su-1))
    data['mu'] = data['mu'].apply(lambda x: np.sum(x,axis=1))**(1 / (su-1)) 
    data['mu'] = data['mu'].apply(lambda x: x.reshape(x.shape[0],1))*(au*((x+1)**(-1 / su)))-c
    data['mu'] = data.apply(lambda x:np.append(x['first'].reshape(1,10),x['mu'][1:],axis=0),axis=1)
    return data['mu']


In [ ]:
n_user=len(ID)


In [ ]:
n_iter = 1000
prg_bar = tqdm(range(n_iter))
for i in prg_bar:
    #au
    #step1
    au_star = sample_au_star(au, (2.4**2/10)*np.identity(10))
    simulate['au_star'] = simulate['ID'].apply(lambda x: au_star[x])

    #step2
    simulate['mu'] = cal_mu(simulate['au'],simulate['su'],simulate['c'],simulate['x'])
    simulate['prob'] = simulate['mu'].apply(lambda x : stable_softmax(x))

    simulate['mu_star'] = cal_mu(simulate['au_star'],simulate['su'],simulate['c'],simulate['x'])
    simulate['prob_star'] = simulate['mu_star'].apply(lambda x : stable_softmax(x))

    simulate['likelihood'] = simulate.apply(lambda x : ((calc_likelihood(x['choice'],x['prob_star']))/(calc_likelihood(x['choice'],x['prob'])))**x['count'],axis=1)

    logN = {key : float((log_normal_pdf(au_star[key], a_mean, a_sigma, n_cate)) / (log_normal_pdf(au[key], a_mean, a_sigma, n_cate))) for key in ID}
    likelihood = simulate.groupby('ID')['likelihood'].prod().to_dict()
    prod = {key : np.prod(au_star[key])/max(np.prod(au[key]),1e-12) for key in ID}

    accept = {key : bool((likelihood[key])*(logN[key])*(prod[key]) > np.random.rand()) for key in ID}
    accept_ratio = np.mean(list(accept.values()))
    #step3
    au = {key: au_star[key]*accept[key] + au[key]*(1-accept[key]) for key in ID}
    simulate['au'] = simulate['ID'].apply(lambda x: au[x])
    accept_a = np.append(accept_a,np.array(list(accept.values()), dtype=bool).reshape(1,n_user),axis=0)
    #step4
    a_mean_bar = (np.asmatrix(np.sum(np.log(list(au.values())),axis=0))*np.linalg.inv(a_sigma)+np.asmatrix(a_mean_bar_init)*np.linalg.inv(a_mean_sigma_init)) \
    *np.linalg.inv(n_user*np.linalg.inv(a_sigma)+np.linalg.inv(a_mean_sigma_init))
    a_mean_sigma = np.linalg.inv(n_user * np.linalg.inv(a_sigma) + np.linalg.inv(a_mean_sigma_init))

    a_sigma_S = np.sum([np.multiply(np.asmatrix(np.log(v)-a_mean),np.transpose(np.asmatrix(np.log(v)-a_mean))) for v in au.values()],axis=0) + a_sigma_S_init
    a_df = a_df_init+n_user
    #step5
    a_mean = multivariate_normal.rvs(np.array(a_mean_bar).reshape(n_cate),a_mean_sigma) 
    a_sigma = invwishartrand_prec(a_df,np.linalg.inv(a_sigma_S))
    
    
    #su
    #step1
    #if i <20 :
    su_star = sample_su_star(su,2**2)
    simulate['su_star'] = simulate['ID'].apply(lambda x: su_star[x])

    #step2
    simulate['mu'] = cal_mu(simulate['au'],simulate['su'],simulate['c'],simulate['x'])
    simulate['mu_star'] = cal_mu(simulate['au'],simulate['su_star'],simulate['c'],simulate['x'])
    simulate['prob'] = simulate['mu'].apply(lambda x : stable_softmax(x))
    simulate['prob_star'] = simulate['mu_star'].apply(lambda x : stable_softmax(x))
    simulate['likelihood'] = simulate.apply(lambda x : ((calc_likelihood(x['choice'],x['prob_star']))/(calc_likelihood(x['choice'],x['prob'])))**x['count'],axis=1)


    logN = {key : float((shifted_log_normal_pdf(su_star[key], s_mean, s_sigma*np.identity(1), 1)) / (shifted_log_normal_pdf(su[key], s_mean, s_sigma*np.identity(1), 1))) for key in ID}
    likelihood = simulate.groupby('ID')['likelihood'].prod().to_dict()
    
    prod = {key : max(su_star[key]-1,1e-12) / max(su[key]-1,1e-12) for key in ID}

    accept = {key : bool((likelihood[key])*(logN[key])*(prod[key]) > np.random.rand()) for key in ID}
    accept_s = np.append(accept_s,np.array(list(accept.values()), dtype=bool).reshape(1,n_user),axis=0)
    #step3
    su = {key: max(float(su_star[key]*accept[key] + su[key]*(1-accept[key])),1+1e-12) for key in ID}
    simulate['su'] = simulate['ID'].apply(lambda x: su[x])
    
    #swi
    #step1
    swi_star = sample_swi_star(swi,2**2)
    simulate['swi_star'] = simulate['ID'].apply(lambda x: swi_star[x])
    simulate['c_star'] = simulate['swi_star']*simulate['switch_cost']

    simulate['mu'] = cal_mu(simulate['au'],simulate['su'],simulate['c'],simulate['x'])
    simulate['prob'] = simulate['mu'].apply(lambda x : stable_softmax(x))

    simulate['mu_star'] = cal_mu(simulate['au'],simulate['su'],simulate['c_star'],simulate['x'])
    simulate['prob_star'] = simulate['mu_star'].apply(lambda x : stable_softmax(x))

    simulate['likelihood'] = simulate.apply(lambda x : ((calc_likelihood(x['choice'],x['prob_star']))/(calc_likelihood(x['choice'],x['prob'])))**x['count'],axis=1)
    logN = {key : float((log_normal_pdf(swi_star[key], swi_mean, swi_sigma*np.identity(1), 1)) / (log_normal_pdf(swi[key], swi_mean, swi_sigma*np.identity(1), 1))) for key in ID}

    likelihood = simulate.groupby('ID')['likelihood'].prod().to_dict()
    prod = {key : swi_star[key] / max(swi[key],1e-12) for key in ID}

    accept = {key : bool((likelihood[key])*(logN[key])*(prod[key]) > np.random.rand()) for key in ID}
    #    accept_ratio = np.mean(list(accept.values()))
        
    accept_swi = np.append(accept_swi,np.array(list(accept.values()), dtype=bool).reshape(1,n_user),axis=0)
    #step3
    swi = {key: max(float(swi_star[key]*accept[key] + swi[key]*(1-accept[key])),1e-12) for key in ID}
    simulate['swi'] = simulate['ID'].apply(lambda x: swi[x])
    simulate['c'] = simulate['swi']*simulate['switch_cost']

    
    #step4
    swi_mean_bar = (np.asmatrix(np.sum(np.log(list(swi.values()))))*np.linalg.inv(swi_sigma*np.identity(1)) + np.asmatrix(swi_mean_bar_init)*np.linalg.inv(swi_mean_sigma_init*np.identity(1))) \
    *(np.linalg.inv(n_user * np.linalg.inv(swi_sigma*np.identity(1)) + np.linalg.inv(swi_mean_sigma_init*np.identity(1))))
    
    swi_mean_sigma = np.linalg.inv(n_user * np.linalg.inv(swi_sigma*np.identity(1)) + np.linalg.inv(swi_mean_sigma_init*np.identity(1)))
    swi_sigma_S = np.sum([np.transpose(np.asmatrix(np.log(v)-swi_mean))*np.asmatrix(np.log(v)-swi_mean) for v in swi.values()],axis=0) + swi_sigma_S_init
    swi_df = swi_df_init+n_user

    #step5
    swi_mean = float(np.random.normal(float(np.asarray(swi_mean_bar).ravel()[0]), float(np.sqrt(np.asarray(swi_mean_sigma).ravel()[0])))) 
    swi_sigma = float(np.asarray(invwishartrand_prec(swi_df,np.linalg.inv(swi_sigma_S))).ravel()[0])
    
    
    simulate['predict'] = simulate['prob'].apply(lambda x : np.asarray(np.argmax(x,axis=1)[:-1]).tolist())
    simulate['similarity'] = simulate.apply(lambda x: difflib.SequenceMatcher(None,x['predict'],x['cat']).ratio(), axis=1)
    
    train_accuracy.append(np.sum(simulate['len']*simulate['similarity']*simulate['count']) / np.sum(simulate['len']*simulate['count']))
    
    #collect parameter
    collect_au = np.append(collect_au,np.array(pd.DataFrame(au)).reshape(1,10,n_user),axis=0)
    collect_a_mean = np.append(collect_a_mean,a_mean.reshape(1,n_cate),axis=0)
    collect_a_sigma = np.append(collect_a_sigma,a_sigma.reshape(1,n_cate,n_cate),axis=0)
    
    collect_su = np.append(collect_su,np.array(pd.DataFrame(su.values()).T),axis=0)
    
    collect_swi = np.append(collect_swi,np.array(pd.DataFrame(swi.values()).T),axis=0)
    collect_swi_mean = np.append(collect_swi_mean,float(swi_mean))
    collect_swi_sigma = np.append(collect_swi_sigma,float(swi_sigma))
    #test acc
    #test_data['mu'] = cal_mu(test_data['ID'],test_data['x'],test_data['choice'],au,su,swi,switching_cost)
    #test_data['predict'] = test_data['mu'].apply(lambda x : np.argmax(x,axis=1)[:-1])
    #test_data['similarity'] = test_data.apply(lambda x: difflib.SequenceMatcher(None,x['predict'],x['cat']).ratio(), axis=1)
    #test_accuary.append(np.sum(test_data['len']*test_data['similarity']) / np.sum(test_data['len']))

    #print('train_acc= %f, test_acc = %f' %(train_accuracy[-1],test_accuary[-1]))
    
    print(np.mean(accept_a,axis=1)[-1],np.mean(accept_s,axis=1)[-1],np.mean(accept_swi,axis=1)[-1])
    print('train_acc= %f' %(train_accuracy[-1]))
    
np.savetxt("au.csv", collect_au.reshape(collect_au.shape[0],n_user*10), delimiter=",")#10000*10*1000
np.savetxt("a_mean.csv", collect_a_mean, delimiter=",")
np.savetxt("a_sigma.csv", collect_a_sigma.reshape(collect_a_sigma.shape[0],n_cate*n_cate), delimiter=",")#iter*10*10

np.savetxt("su.csv", collect_su, delimiter=",")

np.savetxt("swi.csv", collect_swi, delimiter=",")
np.savetxt("swi_mean.csv", collect_swi_mean, delimiter=",")
np.savetxt("swi_sigma.csv", collect_swi_sigma, delimiter=",")

np.savetxt("ID.csv", ID, delimiter=",")
np.savetxt("accuracy.csv", train_accuracy, delimiter =",")



In [ ]:
len(train_accuracy)


In [ ]:
def layout(df,n=24):
    df = df.copy()
    df['mu'] = df['au']**(df['su'] / (df['su']-1))
    return df


In [ ]:
au = {int(key): collect_au[-1,:,:][:,i] for i, key in enumerate(ID)}


In [ ]:
su = {int(key): collect_su[-1,i] for i, key in enumerate(ID)}
swi = {int(key): collect_swi[-1,i] for i, key in enumerate(ID)}


In [ ]:
ID = np.loadtxt('ID.csv',delimiter=',')
ID = np.atleast_1d(ID).astype(int)
n_user = len(ID)

collect_au = np.loadtxt('au.csv',delimiter=',')
collect_au = np.atleast_2d(collect_au).reshape(-1,n_cate,n_user)
collect_a_mean = np.loadtxt('a_mean.csv',delimiter=',')
collect_a_mean = np.atleast_2d(collect_a_mean)
collect_a_sigma = np.loadtxt('a_sigma.csv',delimiter=',')
collect_a_sigma = np.atleast_2d(collect_a_sigma).reshape(-1,n_cate,n_cate)

collect_su = np.loadtxt('su.csv',delimiter=',')
collect_su = np.atleast_2d(collect_su).reshape(-1,n_user)

collect_swi = np.loadtxt('swi.csv',delimiter=',')
collect_swi = np.atleast_2d(collect_swi).reshape(-1,n_user)
collect_swi_mean = np.loadtxt('swi_mean.csv',delimiter=',')
collect_swi_sigma = np.loadtxt('swi_sigma.csv',delimiter=',')

accuracy = np.loadtxt('accuracy.csv',delimiter=',')


In [ ]:
np.mean(accept_swi,axis=1)[-1]


# Converge Diagnostics

In [ ]:
plt.plot(np.mean(accept_swi,axis=1),label='swi')
plt.plot(np.mean(accept_a,axis=1),label='au')
plt.plot(np.mean(accept_s,axis=1),label='s')
plt.legend()


In [ ]:
try:
    simulate = simulate[simulate['ID'].isin(ID)]
except NameError:
    simulate = browsing[browsing['ID'].isin(ID)]


In [ ]:
def geweke_diag(x, frac1=0.1, frac2=0.5):
    len_first = int(x.shape[0]*frac1)
    len_last = int(x.shape[0]*frac2)
    first = x[0:len_first]
    last = x[len_last:]
    Z = (np.mean(first,axis=0)-np.mean(last,axis=0))/((np.std(first,axis=0)**2)/(len_first)+(np.std(last,axis=0)**2)/len_last)**(1/2)
    return Z

def geweke_plot(x, frac1 = 0.1, frac2 = 0.5, nbins = 20, pvalue = 0.05):
    geweke_list = []
    burn_in_list = x.shape[0]*np.arange(0,0.5,0.5/nbins)
    for i in burn_in_list:
        geweke_list.append(geweke_diag(x[int(i):]))
    sns.set(font="sans-serif")
    plt.figure(figsize=(12,8))
    plt.plot(np.arange(0,0.5,0.5/nbins),geweke_list,label='user{}'.format)
    #plt.ylim([-20,20])
    plt.axhline(norm.ppf(pvalue/2),color='r',ls='--')
    plt.axhline(norm.ppf(1-pvalue/2),color='r',ls='--')
    #plt.legend()
    plt.xlabel('burn in ratio')


In [ ]:
geweke_plot(collect_su,nbins=10)


In [ ]:
plt.figure(figsize=(12,8))
for i in range(10):
    plt.plot(np.log(np.mean(collect_au,axis=2))[:,i],label=i)
plt.legend()


In [ ]:
collect_a_mean[:,0]


In [ ]:
collect_a_mean.shape


In [ ]:
collect_au.shape


In [ ]:
np.sum(simulate['x_sum'])


In [ ]:
geweke_list = []
burn_in_list = collect_au.shape[0]*np.arange(0,0.5,0.5/10)
for i in burn_in_list:
    geweke_list.append(geweke_diag(collect_au[int(i):]))


In [ ]:
simulate


## Per-user Inspection


In [ ]:
def inpsepct(cid,au,su,swi,browsing_data=simulate):
   # try :
        index = list(browsing_data['ID'].drop_duplicates()).index(cid)
        print(browsing_data[browsing_data['ID']==cid][['ID','session_id','cat','predict']])
        plt.figure(figsize=(12,8))
        for k in range(10):
            plt.plot(au[:,k,index],label='a{}'.format(k))
        plt.title('au of ID({})'.format(cid))
        plt.legend()
        plt.figure(figsize=(12,8))
        plt.plot(su[:,index],label='su')
        plt.title('su of ID({})'.format(cid))
        plt.legend()
        plt.figure(figsize=(12,8))
        plt.plot(swi[:,index],label='swi')
        plt.title('swi of ID({})'.format(cid))
        plt.legend()
   # except :
   #     print('ID not found')
        


In [ ]:
inpsepct(list(set(simulate['ID']))[10],np.log(collect_au),np.log(collect_su),np.log(collect_swi))


In [ ]:
collect_swi


In [ ]:
# plt.plo  # unfinished scratch cell removed for portfolio run


In [ ]:
def test_converge(parameter, start_iter, end_iter,log=True,n_user=10,seed=1):
    random.seed(seed)
    sns.set(font="sans-serif")
    data = parameter[start_iter:end_iter]
    if log==True:
        data = np.log(data)
    if len(data.shape)==3:
        sample_user = random.sample(range(data.shape[2]), n_user)
        for i in sample_user:
            plt.figure(figsize=(8,6))
            plt.plot(range(start_iter,end_iter),data[:,:,i])
    else:
        sample_user = random.sample(range(data.shape[1]), n_user)
        plt.figure(figsize=(8,6))
        for i in sample_user:
            plt.plot(range(start_iter,end_iter),data[:,i],label='user_{}'.format(i))
        plt.legend()
    return sample_user
#test_converge(collect_au,100,500,0,n_user=10)


In [ ]:
plt.plot(collect_swi_mean)


In [ ]:
L1_dict


In [ ]:
simulate


In [ ]:
# inspect  # unfinished scratch cell removed for portfolio run


In [ ]:
for i in range(10):
    plt.figure(figsize=(12,8))
    sns.distplot(np.mean(np.log(collect_au),axis=0)[i,:])


In [ ]:
most = test_df.set_index('ID')['x_sum'].apply(lambda x : np.argmax(x)).to_dict()
browsing = browsing[browsing['ID'].isin(most.keys())].copy()
browsing['len'] = browsing['cat'].apply(lambda x : len(x))
browsing['most'] = browsing["ID"].apply(lambda x : most[x])
browsing['most'] = browsing['most'].apply(lambda x : [x])
browsing['most'] = browsing.apply(lambda x: x.len*x.most,axis=1)


In [ ]:
browsing['most_acc'] = browsing.apply(lambda x: difflib.SequenceMatcher(None,x['most'],x['cat']).ratio(), axis=1)
np.sum(browsing['len']*browsing['most_acc']) / np.sum(browsing['len'])
